In [ ]:
import torch

In [ ]:
import torch

# ✅ INTERVIEW — 手写 Cross Entropy Loss（不使用 torch.softmax / F.cross_entropy）

def cross_entropy_loss(logits, targets):
    """
    手写实现 Cross Entropy Loss，多分类交叉熵损失。

    参数:
        logits: Tensor, shape = (B, C)
            B 表示 batch size，C 表示类别数。
            logits 是模型输出的原始分数，不是概率。
            例如：一个 3 分类问题，logits[i] = [2.0, 1.0, 0.1] 表示模型认为第 0 类最可能。

        targets: Tensor, shape = (B,)
            每个样本的真实类别索引。
            例如 targets[i] = 2，表示第 i 个样本的真实类别是第 2 类。

    返回:
        一个标量 Tensor，表示 batch 内所有样本交叉熵损失的平均值。
    """

    # ============================================================
    # 数学背景：
    #   交叉熵损失 = -log(softmax(logits)[target])
    #
    #   展开 softmax：
    #   loss = -log(exp(logits[target]) / sum(exp(logits)))
    #
    #   根据 log(a/b) = log(a) - log(b)：
    #   loss = -logits[target] + log(sum(exp(logits)))
    #         ↑ 目标类别的 logit      ↑ log-sum-exp（归一化项）
    #
    #   问题：直接 exp(大 logits) 会溢出！
    #   解决：用 log-sum-exp 技巧，减去最大值再算。
    # ============================================================

    # ------------------------------------------------------------
    # 第一步: 减去最大值，防止 exp 溢出（log-sum-exp 技巧）
    # ------------------------------------------------------------
    # logits 的形状是 (B, C)
    # dim=-1 等价于 dim=1（最后一维，即类别维度）
    # 沿着 dim=-1 取最大值，就是在每一行（每个样本）中找最大的 logit
    #
    # .values 只要值，不要索引（max() 返回 (values, indices) 两个东西）
    # keepdim=True 保持形状为 (B, 1)，方便后面和 (B, C) 的 logits 做广播相减
    #
    # 例：
    # logits =
    # [
    #   [1000., 1001., 1002.],   ← 第 0 个样本
    #   [0.5,   2.5,  0.3 ]     ← 第 1 个样本
    # ]
    #
    # logits.max(dim=-1, keepdim=True).values =
    # [
    #   [1002.],   ← 第 0 行最大值
    #   [2.5  ]    ← 第 1 行最大值
    # ]
    #
    # 为什么要减最大值？
    #   如果直接 exp(1002) → Inf（溢出）
    #   减去最大值后：exp(1002 - 1002) = exp(0) = 1（安全）
    #
    # 数学保证：
    #   softmax(x) == softmax(x - max(x))
    #   因为 exp(x_i - max) / sum(exp(x_j - max))
    #     = (exp(x_i) / exp(max)) / (sum(exp(x_j)) / exp(max))
    #     = exp(x_i) / sum(exp(x_j))
    #   分子分母同时除以 exp(max)，结果不变。
    logits_max = logits.max(dim=-1, keepdim=True).values
    logits_stable = logits - logits_max

    # ------------------------------------------------------------
    # 第二步: 计算 log(sum(exp(logits)))，即 log-partition function
    # ------------------------------------------------------------
    # 这就是公式中的 log(Σ exp(z_j)) 部分
    #
    # 注意：这里用的是 logits_stable（已减去最大值），不是原始 logits
    # torch.exp(logits_stable) → 形状仍然是 (B, C)
    # .sum(dim=-1) → 沿类别维度求和，形状变为 (B,)
    # keepdim=True → 保持形状为 (B, 1)，方便后面广播
    #
    # 为什么必须 keepdim=True？
    #   如果 log_sum_exp 形状是 (B,)，后面 logits_stable - log_sum_exp 时：
    #     logits_stable: (B, C)
    #     log_sum_exp:   (B,)
    #
    #   PyTorch 广播规则是从右往左对齐：
    #     (B, C) 和 (B,) → (B, C) 和 (1, B) → 维度不匹配！❌
    #
    #   用 keepdim=True 让 log_sum_exp 形状为 (B, 1)：
    #     (B, C) - (B, 1) → 广播为 (B, C)，每行都减去该行的 log_sum_exp ✅
    #
    # 例：
    # logits_stable = [[-2., -1., 0.],   ← exp: [0.135, 0.368, 1.0]
    #                   [-2.,  0., -2.2]]  ← exp: [0.135, 1.0, 0.111]
    #
    # exp 之和 = [[1.503], [1.246]]   ← 形状 (2, 1)
    # log_sum_exp = [[0.407], [0.220]]   ← 形状 (2, 1)
    log_sum_exp = torch.log(torch.exp(logits_stable).sum(dim=-1, keepdim=True))

    # ------------------------------------------------------------
    # 第三步: 计算 log-softmax = logits - log(sum(exp(logits)))
    # ------------------------------------------------------------
    # 数学推导：
    #   log(softmax(z_i)) = log(exp(z_i) / Σexp(z_j))
    #                     = z_i - log(Σexp(z_j))
    #                     = z_i - log_sum_exp
    #
    # 这里用 logits_stable 而不是原始 logits：
    #   因为 logits_stable = logits - max，log_sum_exp 也是基于 logits_stable 算的
    #   两者相减后 max 被抵消，结果和用原始 logits 算完全等价
    #
    # 广播：
    #   logits_stable: (B, C) - log_sum_exp: (B, 1) → (B, C)
    #   每一行的每个元素都减去该行的 log_sum_exp
    #
    # 例：
    # logits_stable = [[-2., -1., 0.], [-2., 0., -2.2]]
    # log_sum_exp   = [[0.407],        [0.220]]
    # log_probs     = [[-2.407, -1.407, -0.407],
    #                  [-2.220, -0.220, -2.420]]
    # 验证：exp(-2.407)+exp(-1.407)+exp(-0.407) ≈ 0.09+0.245+0.665 ≈ 1.0 ✅
    log_probs = logits_stable - log_sum_exp

    # ------------------------------------------------------------
    # 第四步: 用高级索引选出每个样本真实类别对应的 log 概率
    # ------------------------------------------------------------
    # 面试高频考点：torch.arange + targets 的高级索引
    #
    # log_probs 形状是 (B, C)，我们需要取出每个样本 targets[i] 对应的那个 log 概率
    #
    # torch.arange(B) 生成行索引 [0, 1, 2, ..., B-1]
    # targets 是列索引（真实类别）
    #
    # log_probs[torch.arange(B), targets] 是「花式索引」（advanced indexing）：
    #   取 (0, targets[0]), (1, targets[1]), ..., (B-1, targets[B-1])
    #
    # 例：
    # log_probs = [[-2.4, -1.4, -0.4],    ← 3 个类别
    #              [-2.2, -0.2, -2.4]]
    # targets = [2, 1]  ← 第 0 个样本真实类别是 2，第 1 个样本真实类别是 1
    #
    # torch.arange(2) = [0, 1]
    # log_probs[[0,1], [2,1]] = [log_probs[0,2], log_probs[1,1]]
    #                           = [-0.4, -0.2]
    #
    # 注意：这里返回的是 (B,) 形状的一维张量，不是 (B, 1)
    B = logits.shape[0]
    target_log_probs = log_probs[torch.arange(B), targets]

    # ------------------------------------------------------------
    # 第五步: 取负均值，得到最终 loss
    # ------------------------------------------------------------
    # 交叉熵定义：loss = -log(p_target)
    # 对 batch 内所有样本取平均：loss.mean()
    #
    # target_log_probs 形状是 (B,)，是每个样本真实类别的 log 概率
    # 取负再求均值 → 标量
    #
    # 例：
    # target_log_probs = [-0.4, -0.2]
    # loss = -mean([-0.4, -0.2]) = -(-0.3) = 0.3
    return -target_log_probs.mean()

In [ ]:
# 验证
logits = torch.randn(4, 10)
targets = torch.randint(0, 10, (4,))
print('手写:', cross_entropy_loss(logits, targets).item())
print('参考:', torch.nn.functional.cross_entropy(logits, targets).item())

In [ ]:
from torch_judge import check
check('cross_entropy')

## 📝 核心思路总结

1. **本质**：交叉熵 loss = `-log(softmax(logits)[target])`，展开后是 `-target_logit + log(Σexp(logits))`，前者是目标类别的原始分数，后者是归一化项（log-partition function）。
2. **数值稳定是核心考点**：直接 `exp(大 logits)` 会溢出到 `Inf`，所以先减去每行最大值（`logits - max(logits)`），这不改变 softmax 结果，但把 exp 的输入压到 ≤ 0，避免溢出。
3. **高级索引是面试必问**：`log_probs[torch.arange(B), targets]` 用行索引 + 列索引同时定位，取出每个样本真实类别对应的 log 概率，这是 PyTorch 取对角线元素的标准技巧。
4. **广播机制要小心**：`logits_stable (B,C) - log_sum_exp (B,1)` 利用广播让每行减去自己的 log-sum-exp；如果 `log_sum_exp` 是 `(B,)` 而不是 `(B,1)`，广播会从右往左对齐导致维度不匹配。
5. **和 `torch.nn.functional.cross_entropy` 等价**：PyTorch 内部也是先 log-softmax 再取目标位置的负对数，只是它把两步 fused 成一个 kernel，数值上完全一致。